# SoccerNet GSR — Colab Inference

Runs the SoccerNet Game State Reconstruction baseline on Colab with GPU.

**Setup:** Runtime → Change runtime type → T4 GPU

**Note:** Jersey number detection (MMOCR) is excluded due to OpenMMLab incompatibility
with Colab's Python 3.12 + torch 2.x environment. All remaining stages work normally.

**Custom video support (Phase 3):** Upload any .mp4 clip — the adapter extracts
frames and creates SoccerNet-compatible metadata automatically.

### Pipeline stages (jersey detection excluded)

| # | Stage | What it does |
|---|-------|-------------|
| 1 | `bbox_detector` | YOLOv11 — detect player & ball bounding boxes |
| 2 | `reid` | PRTReid — extract appearance embeddings + role detection |
| 3 | `track` | StrongSORT + BPBreid — multi-object tracking |
| 4 | `pitch` | NBW calibration — detect pitch lines for homography |
| 5 | `calibration` | NBW calibration — pixel-to-pitch coordinate mapping |
| 6 | `tracklet_agg` | Role-only voting (GK/outfield) — jersey number skipped |
| 7 | `team` | K-means clustering on embeddings → 2 teams |
| 8 | `team_side` | Mean position → which team attacks left/right |


In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"

## Setup — Install dependencies

This cell:
1. Clones our repo + sn-gamestate
2. Patches sn-gamestate's version pins for Colab compatibility (Python 3.12, torch 2.x)
3. Installs all dependencies **except** mmcv/mmdet/mmocr (jersey number detection not available on Colab)

In [ ]:
import os

# === Step 1: Clone repos ===
REPO_URL = "https://github.com/Moiz005/SoccerVision-Player-Tracking-3D-Reconstruction.git"
REPO_NAME = "SoccerVision-Player-Tracking-3D-Reconstruction"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

%cd {REPO_NAME}

if not os.path.exists("sn-gamestate"):
    !git clone https://github.com/SoccerNet/sn-gamestate.git

# === Step 2: Patch sn-gamestate's pyproject.toml ===
# sn-gamestate pins torch==1.13.1 and requires-python>=3.9,<3.10
# Colab has Python 3.12 + torch 2.x — we relax these pins
pyproject_path = "sn-gamestate/pyproject.toml"
with open(pyproject_path, "r") as f:
    content = f.read()

# Allow Python 3.10+
content = content.replace(
    'requires-python = ">=3.9,<3.10"',
    'requires-python = ">=3.9"'
)

# Remove torch version pin (let it use Colab's torch 2.x)
content = content.replace(
    '    "torch==1.13.1",',
    '    "torch",'
)

# Float numpy for Python 3.12 compat
content = content.replace(
    '    "numpy==1.26.4",',
    '    "numpy>=1.26.4",'
)

with open(pyproject_path, "w") as f:
    f.write(content)

print("Patched pyproject.toml: relaxed requires-python, torch pin, numpy pin")

# === Step 3: Install sn-gamestate without resolving deps ===
# --no-deps prevents pip from trying to install torch 1.13.1 or other conflicting versions
%cd sn-gamestate
!pip install --no-deps -e .

# === Step 4: Install git dependencies (skipped by --no-deps) ===
!pip install "prtreid @ git+https://github.com/VlSomers/prtreid"
!pip install "torchreid @ git+https://github.com/VlSomers/bpbreid"

# Install the calibration plugin bundled with sn-gamestate
# NOTE: calibration plugin requires Python <3.10 but works on 3.12; --ignore-requires-python bypasses this
!pip install --no-deps -e plugins/calibration --ignore-requires-python

# === Step 5: Install remaining PyPI deps ===
# NOT installed: mmcv, mmdet, mmocr (no pre-built wheels for Colab Python 3.12 + torch 2.x)
# This means jersey number detection (stage 6) is unavailable
!pip install "tracklab==1.3.24" \
    "soccernet==0.1.55" \
    "lightning==2.0.9" \
    "transformers==4.35.2" \
    "tokenizers==0.15.2" \
    "easyocr==1.7.1"

print("\n=== Dependencies installed ===")
print("Available stages: bbox_detector, reid, track, pitch, calibration, tracklet_agg, team, team_side")
print("Skipped stages:   jersey_number_detect (requires mmcv/mmocr)")

# === Step 6: Compatibility patches ===

# Fix 1: albumentations version conflict (prtreid needs <2.0 for functional import)
# Pin to 1.3.1 to avoid pydantic version clash with tracklab's lightning==2.0.9 (needs pydantic<2.2.0)
!pip install "albumentations==1.3.1" -q
print("albumentations pinned to 1.3.1 (avoids pydantic conflict)")

# Fix 2: prtreid torch.load weights_only issue (PyTorch 2.6+ defaults to True)
TORCHTOOLS = "/usr/local/lib/python3.12/dist-packages/prtreid/utils/torchtools.py"
with open(TORCHTOOLS) as f:
    content = f.read()
content = content.replace(
    "checkpoint = torch.load(fpath, map_location=map_location)",
    "checkpoint = torch.load(fpath, map_location=map_location, weights_only=False)"
)
with open(TORCHTOOLS, "w") as f:
    f.write(content)
print("prtreid torchtools.py patched (weights_only=False)")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

import tracklab
print(f"TrackLab: {getattr(tracklab, '__version__', 'imported OK')}")

import sn_gamestate
print("sn-gamestate: OK")

# Verify mmcv is NOT installed (expected)
try:
    import mmcv
    print(f"mmcv: {mmcv.__version__}")
except ImportError:
    print("mmcv: not installed (expected — jersey detection unavailable)")

print("\nAll core dependencies verified!")

## Upload Custom Video (Phase 3 — Adapter)

Skips the official SoccerNet download and uses your own .mp4 clip instead.

**What this cell does:**
1. Lets you upload a .mp4 file via the Colab file picker
2. Runs `prepare_custom_video()` to extract frames into the SoccerNet-compatible structure
3. Creates `seqinfo.ini`, `gameinfo.ini`, and empty `gt/gt.txt`
4. Sets `CUSTOM_VIDEO_PATH` for the pipeline cell below

The adapter creates this structure:
```
data/custom_video/
  valid/
    {video_name}/
      img1/000001.jpg ...
      seqinfo.ini
      gameinfo.ini
      gt/gt.txt (empty)
```

> **Compatible with:** The pipeline cell below — set `USE_CUSTOM_VIDEO = True` to use your uploaded clip.

> **Compatible with:** The official data download cell at the bottom — skip this cell if you want to run on official SoccerNet data instead.

In [ ]:
import os, sys

# ═══════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit these values for your video
# ═══════════════════════════════════════════════════════════════════════

# Set to True to use your custom video (skip SoccerNet official data)
USE_CUSTOM_VIDEO = True

# Name of your video file (place it in Colab's file sidebar, or use the upload dialog)
VIDEO_FILENAME = "vid_1.mp4"

# Alternative: set a full path (e.g., Google Drive mount)
# VIDEO_PATH = "/content/drive/MyDrive/vid_1.mp4"
# ═══════════════════════════════════════════════════════════════════════

if USE_CUSTOM_VIDEO:
    # === Step 0: Navigate to repo root (in case install cell wasn't run yet) ===
    REPO_NAME = "SoccerVision-Player-Tracking-3D-Reconstruction"
    if os.path.exists(REPO_NAME):
        %cd $REPO_NAME
    elif os.path.basename(os.getcwd()) != REPO_NAME:
        # Try parent directory
        parent = os.path.dirname(os.getcwd())
        if os.path.basename(parent) == REPO_NAME:
            %cd $parent

    print(f"Working directory: {os.getcwd()}")

    # === Step 1: Locate the video file ===
    # Priority: explicit VIDEO_PATH > file already on disk > upload dialog
    if 'VIDEO_PATH' in dir() and VIDEO_PATH and os.path.exists(VIDEO_PATH):
        video_path = VIDEO_PATH
        video_filename = os.path.basename(video_path)
        print(f"Using video from path: {video_path}")
    elif os.path.exists(os.path.join(os.getcwd(), VIDEO_FILENAME)):
        # File already in Colab's filesystem (drag-drop sidebar, wget, etc.)
        video_path = os.path.join(os.getcwd(), VIDEO_FILENAME)
        video_filename = VIDEO_FILENAME
        file_size = os.path.getsize(video_path)
        print(f"Found existing file: {video_path} ({file_size / 1e6:.1f} MB)")
    else:
        # File not found — show upload dialog
        from google.colab import files
        print(f"'{VIDEO_FILENAME}' not found. Please upload your .mp4 clip...")
        uploaded = files.upload()  # dict: {filename: bytes}
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        video_filename = list(uploaded.keys())[0]
        video_path = os.path.join(os.getcwd(), video_filename)
        print(f"Saved: {video_path} ({len(uploaded[video_filename]) / 1e6:.1f} MB)")

    if not video_filename.lower().endswith(".mp4"):
        print(f"Warning: '{video_filename}' is not an .mp4. Will attempt anyway.")

    # === Step 2: Import and run the adapter ===
    sys.path.insert(0, os.getcwd())
    from src.inference.run_gsr import prepare_custom_video

    CUSTOM_DATA_DIR = os.path.join(os.getcwd(), "data", "custom_video")
    video_name = os.path.splitext(video_filename)[0]

    print(f"\nPreparing custom video: {video_name}")
    meta = prepare_custom_video(video_path, CUSTOM_DATA_DIR, video_name)

    print(f"\n✅ Adapter complete!")
    print(f"   Frames extracted: {meta['frame_count']}")
    print(f"   FPS:              {meta['fps']}")
    print(f"   Resolution:       {meta['width']}x{meta['height']}")
    print(f"   Output path:      {meta['output_path']}")

    # === Step 3: Set path for pipeline cell ===
    CUSTOM_VIDEO_PATH = CUSTOM_DATA_DIR
    print(f"\nSet CUSTOM_VIDEO_PATH = {CUSTOM_VIDEO_PATH}")
    print("Now go to the pipeline cell — it already has USE_CUSTOM_VIDEO = True (set above).")
else:
    print("USE_CUSTOM_VIDEO is False. Set it to True and re-run to use your video.")

## Download Dataset

> **Skip this section if you're using a custom video.** The Upload Custom Video cell above
> already prepared your clip. Jump straight to **Run Baseline — Official or Custom Video**.

Downloads the SoccerNet GSR validation split (~11GB). This only needs to run once — Colab persists files within a session.

Dataset structure after download + restructuring:
```
data/SoccerNetGS/
  valid/
    SNGS-021/
      img1/          # Frame images (000001.jpg, 000002.jpg, ...)
      Labels-GameState.json
    SNGS-022/
      ...
  gamestate-2024/    # Zip files (extracted + restructured above)
```

In [ ]:
import os, zipfile, glob, shutil

DATA_DIR = f"/content/{REPO_NAME}/sn-gamestate/data/SoccerNetGS"
os.makedirs(DATA_DIR, exist_ok=True)

# Download validation split
from SoccerNet.Downloader import SoccerNetDownloader
dl = SoccerNetDownloader(LocalDirectory=DATA_DIR)
dl.downloadDataTask(task="gamestate-2024", split=["valid"])

# Extract all zip files (SoccerNet places them in gamestate-2024/ subdirectory)
zips = glob.glob(os.path.join(DATA_DIR, "**/*.zip"), recursive=True)
for zip_path in zips:
    print(f"Extracting {os.path.basename(zip_path)}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)
    print(f"  Done")

# Restructure: pipeline expects valid/SNGS-XXX/img1/ but zip extracts into gamestate-2024/SNGS-XXX/img1/
gs_dir = os.path.join(DATA_DIR, "gamestate-2024")
if os.path.exists(gs_dir):
    valid_dir = os.path.join(DATA_DIR, "valid")
    os.makedirs(valid_dir, exist_ok=True)
    videos = sorted([d for d in os.listdir(gs_dir) if d.startswith("SNGS-")])
    for v in videos:
        src = os.path.join(gs_dir, v)
        dst = os.path.join(valid_dir, v)
        if not os.path.exists(dst):
            shutil.move(src, dst)
            print(f"  Moved {v} -> valid/{v}")
    print(f"\nDataset ready: {len(videos)} videos in valid/")
    # Show structure of first video
    if videos:
        sample = os.path.join(valid_dir, videos[0])
        print(f"  Example: valid/{videos[0]}/")
        for item in os.listdir(sample):
            path = os.path.join(sample, item)
            if os.path.isdir(path):
                print(f"    {item}/ ({len(os.listdir(path))} files)")
            else:
                print(f"    {item}")
else:
    print("\ngamestate-2024/ folder not found - checking for flat SNGS-XXX folders...")
flat_videos = sorted([d for d in os.listdir(DATA_DIR) if d.startswith("SNGS-")])
if flat_videos:
    valid_dir = os.path.join(DATA_DIR, "valid")
    os.makedirs(valid_dir, exist_ok=True)
    for v in flat_videos:
        src = os.path.join(DATA_DIR, v)
        dst = os.path.join(valid_dir, v)
        if not os.path.exists(dst):
            shutil.move(src, dst)
            print(f"  Moved {v} -> valid/{v}")
    print(f"\nDataset ready: {len(flat_videos)} videos in valid/")
else:
    print("WARNING: No SNGS-XXX folders found. Check the download.")

## Run Baseline — Official or Custom Video

Runs the pipeline on either:
- **Official SoccerNet data** (after running the Download Dataset cell)
- **Your custom .mp4 clip** (after running the Upload Custom Video cell with `USE_CUSTOM_VIDEO = True`)

The pipeline executes 8 stages (jersey number detection excluded):
1. **bbox_detector** — YOLOv11 detects players + ball
2. **reid** — PRTReid extracts appearance embeddings
3. **track** — StrongSORT tracks players across frames
4. **pitch** — NBW calibration finds pitch lines
5. **calibration** — Maps pixels to real-world pitch coordinates
6. **tracklet_agg** — Aggregates tracklets, assigns roles (GK/outfield)
7. **team** — K-means clusters players into 2 teams
8. **team_side** — Determines left/right attacking direction

> **Important:** The default `tracklet_agg` config (`voting_role_jn`) expects `jersey_number_detection` which we skip.
> This cell creates `voting_role_only.yaml` (attributes: ["role"]) and uses it instead.

**Controls:**
- `USE_CUSTOM_VIDEO = False` → runs on official SoccerNet data (uses `DATASET_PATH`)
- `USE_CUSTOM_VIDEO = True`  → runs on your uploaded clip (uses `CUSTOM_VIDEO_PATH`)
- `dataset.nframes=4` → first 4 frames only; set to `-1` for full video

Model weights auto-download on first run (~2GB).

In [ ]:
import os, subprocess, shutil, yaml

# ── Absolute path to the sn-gamestate repo ───────────────────────────
REPO_NAME = "SoccerVision-Player-Tracking-3D-Reconstruction"
SN_GS_DIR = f"/content/{REPO_NAME}/sn-gamestate"
# ────────────────────────────────────────────────────────────────────

# ── Data source — matches the Upload Custom Video cell above ────────
USE_CUSTOM_VIDEO = True    # ← Set True for custom video, False for official SoccerNet
# ────────────────────────────────────────────────────────────────────

if USE_CUSTOM_VIDEO:
    # Custom video path (set by the upload cell)
    DATASET_PATH = CUSTOM_VIDEO_PATH
else:
    # Official SoccerNet validation data path
    DATASET_PATH = f"{SN_GS_DIR}/data/SoccerNetGS"

print(f"Dataset path: {DATASET_PATH}")
print(f"Exists: {os.path.exists(DATASET_PATH)}")

# Locate the tracklab CLI
tracklab_cmd = shutil.which("tracklab")
print(f"tracklab binary: {tracklab_cmd}")

# === Step 1: Create voting_role_only config ===
# The default tracklet_agg config (voting_role_jn.yaml) declares
# attributes: ["jersey_number", "role"]. Since jersey_number_detect
# is not available (no mmocr), we create a role-only variant.
# MajorityVoteTracklet will then require only role_detection +
# role_confidence (produced by PRTReId stage 2).
#
# IMPORTANT: We use absolute paths so this works regardless of cwd.
# The file must go in sn-gamestate's configs dir (where --config-dir points).
TRACKLET_AGG_DIR = os.path.join(SN_GS_DIR, "sn_gamestate", "configs", "modules", "tracklet_agg")
TRACKLET_AGG_CFG = os.path.join(TRACKLET_AGG_DIR, "voting_role_only.yaml")
os.makedirs(TRACKLET_AGG_DIR, exist_ok=True)
with open(TRACKLET_AGG_CFG, "w") as f:
    yaml.dump({
        "_target_": "tracklab.wrappers.MajorityVoteTracklet",
        "cfg": {"attributes": ["role"]}
    }, f)
print(f"Created {TRACKLET_AGG_CFG} (attributes: [role])")

# === Step 2: Run 8-stage pipeline ===
# Overrides:
#   ~modules.jersey_number_detect  = remove MMOCR module from config
#   modules/tracklet_agg=voting_role_only = use role-only aggregation
#   pipeline=... = explicit 8-stage list (jersey not included)
#   dataset.nframes=4 = first 4 frames only (set -1 for full video)
#   dataset.dataset_path = points to custom_video (with valid/ subdir)
result = subprocess.run(
    [tracklab_cmd, "-cn", "soccernet",
     "--config-dir", os.path.join(SN_GS_DIR, "sn_gamestate", "configs"),
     "~modules.jersey_number_detect",
     "modules/tracklet_agg=voting_role_only",
     "pipeline=[bbox_detector,reid,track,pitch,calibration,tracklet_agg,team,team_side]",
     f"dataset.dataset_path={DATASET_PATH}",
     "dataset.nframes=4"],
    capture_output=True, text=True, timeout=600
)

print("\n" + "="*60)
print("STDOUT:")
print("="*60)
print(result.stdout)
print("\n" + "="*60)
print("STDERR (last 5000 chars):")
print("="*60)
print(result.stderr[-5000:] if len(result.stderr) > 5000 else result.stderr)
print("\n" + "="*60)
print(f"Return code: {result.returncode}")
print("="*60)

## View Results

The pipeline generates:
- **Annotated video** — bounding boxes, player IDs, pitch overlay
- **Tracker state** (.pklz) — full tracking data for re-analysis
- **Evaluation metrics** — GS-HOTA score (official SoccerNet benchmark)

In [ ]:
import glob
from IPython.display import Video, display

output_videos = glob.glob(
    "outputs/**/visualization/videos/*.mp4",
    recursive=True
)

print(f"Found {len(output_videos)} output video(s):")
for v in output_videos:
    print(f"  - {v}")

if output_videos:
    print(f"\nPlaying: {output_videos[0]}")
    display(Video(output_videos[0], width=800))
else:
    print("No output videos found. Check the pipeline output above for errors.")

## Next Steps

If this ran successfully, the baseline works on official data.

**Run on your own clip:**
1. Scroll up to the **Upload Custom Video** section
2. Set `USE_CUSTOM_VIDEO = True` and upload your .mp4
3. In the pipeline cell, also set `USE_CUSTOM_VIDEO = True`
4. Run the pipeline cell → it will process your uploaded clip

**Upcoming features (Phases 5-8):**
- Output parsing → JSON/CSV export of tracking data
- Debug overlay video with bounding boxes + team colors
- Tactical minimap (top-down 2D view of player positions)